# LoanSense - Final Pipeline

In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

## 1. Load Data

In [2]:
df = pd.read_csv("loan_approval_data.csv")

## 2. Feature Engineering
- Created `DTI_Ratio_sq` and `Credit_Score_sq` as squared features
- Based on analysis in `01_LoanSense_Analysis.ipynb`

In [3]:
df["DTI_Ratio_sq"] = df["DTI_Ratio"] ** 2
df["Credit_Score_sq"] = df["Credit_Score"] ** 2

## 3. Input - Output Split

In [4]:
# Input-Output

X = df.drop(columns = ["Loan_Approved", "DTI_Ratio", "Credit_Score"])
y = df["Loan_Approved"].map({"Yes" : 1, "No" : 0})

imputer = SimpleImputer(strategy = "most_frequent")
y = imputer.fit_transform(y.values.reshape(-1, 1)).ravel().astype(int)

In [5]:
# Numerical & Categorical Columns 

num_features = X.select_dtypes(include = ["float64", "int64"]).columns
cat_features = X.select_dtypes(include = "object").columns

## 4. Train Test Split

In [6]:
# Train Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42, stratify = y
)

## 5. Pipeline
Best parameters found from analysis:
- `max_depth` = 9
- `min_samples_split` = 10

In [7]:
# Preprocessing Pipeline

preprocessor = ColumnTransformer(
    transformers = [
        ("num", Pipeline(
            steps = [
                ("imputer", SimpleImputer(strategy = "mean")),
                 ("scaler", StandardScaler())
                 ]), num_features),
        ("cat", Pipeline(
            steps = [
                ("imputer", SimpleImputer(strategy = "most_frequent")),
                ("encoder", OneHotEncoder(drop = "first", handle_unknown = "ignore", sparse_output = False))
                 ]), cat_features)
            ]
        )

In [8]:
# Model 

model = DecisionTreeClassifier(
    max_depth = 9,
    min_samples_split = 10
)

In [9]:
# Pipeline

pipe = Pipeline(
    steps = [
        ("Preprocess", preprocessor),
        ("Model", model)
    ]
)

pipe.fit(X_train, y_train)

Pipeline(steps=[('Preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Applicant_ID', 'Applicant_Income', 'Coapplicant_Income', 'Age',
       'Dependents', 'Existing_Loans', 'Savings', 'Collateral_Value',
       'Loan_Amount', 'Loan_Term', 'DTI_Ratio_sq', 'Credit_Score_sq'],
      dtype='object')),
                                                 ('c...
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  Index(['Employment_Status', 'Marital_Status', 'Loan_Purpose', 'Property_Area',
       'Education_Level', 'Gender', 'Employer_Category'],
      dtype='object'))])),
                ('Model',
                 DecisionTreeClassifier(max_depth=9, min_samples_split=10))])

## 6. Results

In [10]:
# Evaluation

y_pred = pipe.predict(X_test)

print("Precision : ", precision_score(y_test, y_pred) * 100, "%")
print("Accuracy : ", accuracy_score(y_test, y_pred) * 100, "%")
print("Recall : ", recall_score(y_test, y_pred) * 100, "%")
print("F1 Score : ", f1_score(y_test, y_pred) * 100, "%")
print("CM : ", confusion_matrix(y_test, y_pred))

Precision :  84.375 %
Accuracy :  92.0 %
Recall :  90.0 %
F1 Score :  87.09677419354838 %
CM :  [[130  10]
 [  6  54]]


## 7. Summary
- Final Precision Score: 84.375 %
- Final Accuracy Score: 92.0 %
- Model: Decision Tree Classifier